# 실습 8: 기준 모델과 추천 모델
- 상황: 아무것도 안 해도 93점이 나온다는 걸 알았다
- 목표: 비교할 기준을 먼저 만들고, 그 위에서 진짜 모델을 재본다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 불러오기
df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

# 2. 센서 열 빈칸을 중앙값으로 채우기
sensor_cols = [c for c in df.columns if c != "result"]
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

# 3. 불량여부 열 만들기
df["불량여부"] = (df["result"] == "불량").astype(int)

# 4. X, y 나누기
X = df[sensor_cols]
y = df["불량여부"]

# 5. 학습용·시험용 나누기
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train 행 수:", len(X_train), "/ 불량 건수:", (y_train == 1).sum())
print("X_test 행 수:", len(X_test), "/ 불량 건수:", (y_test == 1).sum())


X_train 행 수: 1253 / 불량 건수: 83
X_test 행 수: 314 / 불량 건수: 21


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 비교할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 기준 모델 | 학습을 전혀 하지 않고 늘 같은 답만 내놓는 모델. 비교의 바닥선이 된다 |
| 학습 | 답이 붙은 기록을 넣어 규칙을 찾게 하는 일 |
| 예측 | 처음 보는 기록에 답을 붙이는 일 |
| 정확도 | 전체 중 맞힌 비율. 오늘 쓰는 유일한 점수이고, 내일 이 점수를 의심하게 된다 |

## Step 2. 게으름뱅이 모델 만들기

In [2]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(양품)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.zeros(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델이 불량이라 한 건수:", 기준예측.sum())
print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")

기준 모델이 불량이라 한 건수: 0
기준 모델 정확도: 93.31 %


학습이라는 걸 아예 하지 않았습니다. 데이터를 보지도 않았어요. 그냥 전부 0이라고 답한 게 전부입니다. 그런데 93점입니다.

In [3]:
# 시험용에서 양품이 몇 건, 불량이 몇 건인지
print("시험용 양품:", (y_test == 0).sum(), "건")
print("시험용 불량:", (y_test == 1).sum(), "건")

# 전부 양품이라 답하면 -> 양품은 다 맞고, 불량은 다 틀린다
print("맞힌 것:", (y_test == 0).sum(), "/", len(y_test))

시험용 양품: 293 건
시험용 불량: 21 건
맞힌 것: 293 / 314


[기준 모델이 높은 점수를 받는 이유]<br>
시험용 [314]건 중 양품이 [293]건이다.<br>
전부 양품이라 답하면 [293]건은 자동으로 맞는다.<br>
불량 [21]건은 전부 놓치지만, 개수가 적어 점수에 거의 영향이 없다.

## Step 4. 모델을 추천받기

1. 의사결정나무 (Decision Tree Classifier)
if-then 규칙으로 어떤 센서가 어떤 기준값을 넘어서 불량으로 갈렸는지 그대로 보여줄 수 있어 설명이 쉽고, 열마다 자릿수가 달라도 스케일에 영향을 안 받아 단위를 맞출 필요가 없다.

2. 로지스틱 회귀 (Logistic Regression)
각 센서마다 계수(coefficient) 하나씩 나와서 "이 값이 커질수록 불량 쪽으로 얼마나 기우는지"를 숫자로 바로 설명할 수 있다. — 단, 이 모델은 단위를 맞춰야 해. 열마다 자릿수가 제각각인 채로 넣으면 계수 크기가 단위 차이 때문에 왜곡되고(어느 센서가 진짜 중요한지 못 읽음), 수렴도 잘 안 될 수 있어 표준화(StandardScaler 등)를 먼저 해줘야 한다.

둘 다 "기본 상태"로 오늘 조건에 맞고, 불균형 보정 설정은 넣지 않았다.

[추천받은 모델]<br>
1. [로지스틱 회귀] - [둘 중 하나를 고르는 문제의 기본이고, 어느 열이 얼마나 작용했는지 볼 수 있다]<br>
2. [의사결정나무] - [자르는 기준이 눈에 보여서 설명하기 쉽다]<br>
내가 고른 것 : [로지스틱 회귀]

## Step 5. 로지스틱 회귀 학습시키고 점수 재기
단위를 맞춰야 하는 모델이라 표준화(StandardScaler)를 함께 넣는다. 불균형 보정 설정 없이, 기본 상태로 학습한다.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 로지스틱 회귀는 단위를 맞춰야 하므로 표준화한다
# X_train 기준으로 fit하고, X_test는 그 기준 그대로 transform만 한다
scaler = StandardScaler()
X_train_스케일 = scaler.fit_transform(X_train)
X_test_스케일 = scaler.transform(X_test)

# 불균형 보정 설정 없이, 기본 상태로 학습한다
model = LogisticRegression()
model.fit(X_train_스케일, y_train)

예측 = model.predict(X_test_스케일)

정확도 = accuracy_score(y_test, 예측) * 100
불량예측건수 = (예측 == 1).sum()
그중실제불량 = ((예측 == 1) & (y_test == 1)).sum()

print("1) 정확도:", round(정확도, 2), "%")
print("2) 불량이라고 예측한 건수:", 불량예측건수, "건")
print("3) 그중 실제로 불량이었던 건수:", 그중실제불량, "건")


1) 정확도: 93.95 %
2) 불량이라고 예측한 건수: 2 건
3) 그중 실제로 불량이었던 건수: 2 건


### 결과 정리 (실행 결과 기준)

1. 정확도: **93.95%**
2. 불량이라고 예측한 건수: **2건**
3. 그중 실제로 불량이었던 건수: **2건**

기준 모델(전부 양품, 93.31%)보다 정확도는 조금 더 높아졌고(93.95%), 이번엔 아예 안 찍던 불량을 2건 찍었는데 둘 다 맞혔다. 다만 실제 불량 21건 중 19건은 여전히 놓쳤다 — 정확도는 올랐지만 불량을 잡아내는 데는 아직 크게 못 미친다.

## Step 6. 모델 기록표

| 모델 | 왜 썼나 | 정확도 | 불량이라 한 건수 | 그중 진짜 |
|---|---|---|---|---|
| 기준 모델 (전부 양품) | 비교할 바닥선 | [93.31]% | [0] | [0] |
| [로지스틱 회귀] | [분류의 기본이고 결과를 설명하기 쉬워서] | [93.95]% | [2] | [2] |